# RAG Policy Assistant — sandbox/RAG.ipynb

Modular sandbox for building, benchmarking, and iterating on the Meridian Health Group benefits RAG pipeline.  
**All experiments happen here. Nothing moves to `starter/rag_starter.py` until explicitly approved.**

| Step | What you build | Why it matters |
|------|----------------|----------------|
| 1 | Load Documents | Read `data/benefits_policies.md` as raw text |
| 2 | Chunk | Split on `## POL-NNN` boundaries — preserves headers for citation |
| 3 | Embed | `all-MiniLM-L6-v2` — free, local, CPU, ~80 MB one-time download |
| 4 | Retrieve | Cosine similarity over numpy vectors; returns top-k policy chunks |
| 5 | Answer | Claude returns structured JSON via tool use; cites POL-NNN; tags constraints |
| 6 | Evaluate | Batch runner over all 15 sample questions; stores results for Step 7 |
| 7 | Test | Assert correctness against TEST_SPEC; print PASS/FAIL + metrics |

> Run each cell top-to-bottom with **Shift+Enter**.

---

## Environment Setup

Credentials are loaded from a `.env` file. Required keys: `PORTKEY_API_KEY` and `MY_MODEL`. **Never commit `.env`.**

In [743]:
%pip install anthropic python-dotenv sentence-transformers numpy -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Initialize clients

Anthropic client via Portkey gateway + `sentence-transformers` embedding model loaded once at startup.

In [744]:
import os
import csv
import json
import re
import time
import numpy as np
from collections import Counter
from pathlib import Path
import anthropic
from anthropic import Anthropic
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

# Pre-flight: fail fast with a clear message if credentials are missing
_required = ["PORTKEY_API_KEY", "MY_MODEL"]
_missing = [k for k in _required if not os.environ.get(k)]
if _missing:
    raise EnvironmentError(f"Missing required env vars: {', '.join(_missing)}. Check your .env file.")

# client initialization with Portkey gateway
client = Anthropic(
    api_key=os.environ["PORTKEY_API_KEY"],
    base_url="https://portkeygateway.perficient.com",
    default_headers={"x-portkey-api-key": os.environ["PORTKEY_API_KEY"]}
)


### Constants

In [745]:
# model name is set in the .env file (e.g., "@aws-bedrock-use2/us.anthropic.claude-sonnet-4-6")
MODEL = os.environ["MY_MODEL"]

# directory for storing data files (policies, questions, etc.)
DATA_DIR = Path("../data")
POLICIES = DATA_DIR / "benefits_policies.md"
QUESTIONS_CSV = DATA_DIR / "sample_questions.csv"
BATCH_RESULTS_PATH = DATA_DIR / "batch_results.json"

# max number of characters in a question (to avoid hitting the model's context window limit)
MAX_QUESTION_LEN = 500

# max number of retries for the answer function (to handle transient errors)
MAX_RETRIES = 3

### Expected outcomes from questions and policies

Single source of truth for Step 7 assertions. `min_constraints` marks only tags that **must** appear (asserted by the test); extra tags emitted by the model are acceptable. The Explanation column documents the policy rationale.

† = constraint is expected from the model but **not asserted** in `min_constraints` — model's current behavior is defensible.

| Q | answer_type | Expected constraints | Required policies | Explanation |
|---|-------------|---------------------|-------------------|-------------|
| Q01 | answered | `derived` | POL-002 | QLE changes must be submitted within 30 days; marriage triggers the window (POL-002). |
| Q02 | answered | `conflict` | POL-007, POL-015 | POL-015 supersedes POL-007 match/vesting effective 2026-01-01; both must be surfaced. |
| Q03 | answered | `other` | POL-010 | Parental leave weeks fully stated in POL-010; model tags `other` for the concurrent FMLA reference. |
| Q04 | answered | `compound_eligibility` | POL-005, POL-006 | POL-006 bars Healthcare FSA + HSA in the same plan year; both policies required. |
| Q05 | answered | `compound_eligibility` | POL-001 | 24 hrs/week = part-time; POL-001 gates benefits by employment class. |
| Q06 | answered | `derived` † | POL-008 | Three PTO tiers by tenure; 7 years falls in mid-tier; carryover cap stated. Model currently tags `[]` — tier lookup without explicit arithmetic. |
| Q07 | answered | `compound_eligibility` | POL-011 | Remote-work stipend explicitly excludes on-site-only employees (POL-011). |
| Q08 | answered | — | POL-003 | Silver PPO deductible is a direct lookup in POL-003. |
| Q09 | answered | — | POL-004 | Orthodontia lifetime maximum directly stated in POL-004. |
| Q10 | answered | `other` | POL-014 | Cloud certification reimbursement covered by POL-014; model tags `other` for the pre-approval requirement. |
| Q11 | answered | `conflict` | POL-007, POL-015 | Future-tense framing (next year = 2026); POL-015 supersedes POL-007 effective 2026-01-01. |
| Q12 | answered | `conflict` | POL-016 | POL-016 states eligibility after 5 years but approval requires 7 years; both sides must be reported. |
| Q13 | **declined** | `incomplete` | — | Neither POL-008 nor POL-010 addresses PTO accrual during parental leave. |
| Q14 | answered | `incomplete` | POL-017 | First bereavement request needs no documentation; subsequent requests unaddressed — clear coverage gap. |
| Q15 | answered | — | POL-005 | HSA seed for family coverage on Bronze HDHP is directly stated in POL-005. |

In [746]:
QUESTION_SPEC = {
    "Q01": {"question": "I just got married last month - can I add my spouse to my medical plan now?",
            "answer_type": "answered", "min_constraints": {"derived"}, "min_policies": {"POL-002"}},
    "Q02": {"question": "How much does the company match on my 401k and when does it vest?",
            "answer_type": "answered", "min_constraints": {"conflict"}, "min_policies": {"POL-007", "POL-015"}},
    "Q03": {"question": "I'm having a baby in the spring - how many weeks of paid parental leave do I get?",
            "answer_type": "answered", "min_constraints": set(), "min_policies": {"POL-010"}},
    "Q04": {"question": "Can I have both an HSA and a healthcare FSA at the same time?",
            "answer_type": "answered", "min_constraints": set(), "min_policies": {"POL-005", "POL-006"}},
    "Q05": {"question": "I work 24 hours a week - which benefits am I eligible for?",
            "answer_type": "answered", "min_constraints": {"compound_eligibility"}, "min_policies": {"POL-001"}},
    "Q06": {"question": "How many PTO days do I accrue after 7 years here and how much carries over?",
            "answer_type": "answered", "min_constraints": set(), "min_policies": {"POL-008"}},
    "Q07": {"question": "Does the company help pay for my home internet if I work remotely?",
            "answer_type": "answered", "min_constraints": set(), "min_policies": {"POL-011"}},
    "Q08": {"question": "What is the deductible on the Silver PPO plan?",
            "answer_type": "answered", "min_constraints": set(), "min_policies": {"POL-003"}},
    "Q09": {"question": "Is there a lifetime maximum for my kid's braces under the dental plan?",
            "answer_type": "answered", "min_constraints": set(), "min_policies": {"POL-004"}},
    "Q10": {"question": "Can I get reimbursed for a cloud certification course?",
            "answer_type": "answered", "min_constraints": set(), "min_policies": {"POL-014"}},
    "Q11": {"question": "What will the 401k match and vesting schedule be starting next year?",
            "answer_type": "answered", "min_constraints": {"conflict"}, "min_policies": {"POL-007", "POL-015"}},
    "Q12": {"question": "How many years of service do I need before I can take a sabbatical?",
            "answer_type": "answered", "min_constraints": {"conflict"}, "min_policies": {"POL-016"}},
    "Q13": {"question": "If I take parental leave, does my PTO keep accruing while I'm out?",
            "answer_type": "declined", "min_constraints": {"incomplete"}, "min_policies": set()},
    "Q14": {"question": "My grandfather passed away - how much bereavement leave do I get?",
            "answer_type": "answered", "min_constraints": {"incomplete"}, "min_policies": {"POL-017"}},
    "Q15": {"question": "I'm on the Bronze HDHP - how much does the company put into my HSA for family coverage?",
            "answer_type": "answered", "min_constraints": set(), "min_policies": {"POL-005"}},
}

# Derived views — edit QUESTION_SPEC above; do not edit these directly
RECALL_GT = {
    qid: (spec["question"], spec["min_policies"])
    for qid, spec in QUESTION_SPEC.items()
    if spec["min_policies"]   # Q13 excluded — no ground-truth policies
}

TEST_SPEC = {
    qid: {k: spec[k] for k in ("answer_type", "min_constraints", "min_policies")}
    for qid, spec in QUESTION_SPEC.items()
}

### Util functions

In [747]:
# ── Model loading ─────────────────────────────────────────────────────────────
def _load_model(model_name):
    """Load a SentenceTransformer from local cache; download only on first run."""
    try:
        return SentenceTransformer(model_name, local_files_only=True)
    except Exception:
        print(f"Cache miss — downloading {model_name} (one-time download)...")
        return SentenceTransformer(model_name)

# ── Input validation ───────────────────────────────────────────────────────────
def _validate_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must be a non-empty string.")
    if len(question) > MAX_QUESTION_LEN:
        raise ValueError(f"Question exceeds {MAX_QUESTION_LEN}-character limit ({len(question)} chars).")
    return question.strip()

---
## Step 1: Load Documents

Read `data/benefits_policies.md` as raw text. Maps to `load_documents()` in `rag_starter.py`.

In [748]:
def load_documents(path=POLICIES):
    text = path.read_text(encoding="utf-8")
    print(f"Loaded {len(text):,} characters")
    print(text[:300])
    return text

text = load_documents()

Loaded 6,423 characters
# Meridian Health Group - Employee Benefits & HR Policy Handbook (Synthetic)

> Internal reference for the People Operations team. This is a SYNTHETIC document
> created for training. All plan names, numbers, and vendors are fictional.
>
> **Handbook edition: current as of 1 October 2025.** Treat th


## Step 2: Chunk

Split the handbook text into retrievable passages. Three strategies are compared and benchmarked in **Step 2.1**; the selected strategy is implemented in **Step 2.2**.

Maps to `chunk()` in `rag_starter.py`.

### Step 2.1: Benchmark Chunks

Compare three chunking strategies on this dataset before committing to one. Evaluation is purely embedding-based — no LLM calls.

### Strategies

| # | Strategy | How it works |
|---|----------|-------------|
| 1 | **Policy-boundary** *(current)* | Split on `## POL-NNN` markers — one chunk per policy, header always present in every chunk |
| 2 | **Sliding window** | Word-count-based fixed windows (~200 words, 50-word overlap). Ignores policy boundaries — content from adjacent policies may merge into one chunk |
| 3 | **Paragraph + header injection** | Split each policy at double-newline paragraph breaks; re-prepend the `## POL-NNN` header to every sub-chunk so citation identity is preserved even when a policy is divided |

### Metrics

| Metric | What it measures |
|--------|-----------------|
| `n_chunks` | Total chunk count — fewer = less retrieval noise per query |
| `header_coverage` | % of chunks containing a `## POL-` header — citation readiness |
| `recall@3` | For 14 questions with known ground-truth policies: fraction where ALL required policy chunks appear in the top-3 retrieved results |
| `compound_recall@3` | Same, restricted to the 3 questions that require 2 policies simultaneously (Q02, Q04, Q11) |

In [749]:
# ── Strategy 1: policy-boundary (current) ────────────────────────────────────
def chunk_policy(text):
    """One chunk per ## POL-NNN policy block. Header always present."""
    parts = re.split(r"(?=^## POL-)", text, flags=re.MULTILINE)
    return [p.strip() for p in parts if p.strip().startswith("## POL-")]

# ── Strategy 2: sliding window ────────────────────────────────────────────────
def chunk_sliding(text, size=200, overlap=50):
    """Fixed word-count windows; does not respect policy boundaries."""
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += size - overlap
    return chunks

# ── Strategy 3: paragraph + header injection ─────────────────────────────────
def chunk_paragraph(text):
    """Split each policy at paragraph breaks; re-inject POL header into every sub-chunk."""
    policies = re.split(r"(?=^## POL-)", text, flags=re.MULTILINE)
    chunks = []
    for pol in policies:
        pol = pol.strip()
        if not pol.startswith("## POL-"):
            continue
        lines = pol.split("\n")
        header = lines[0]
        body = "\n".join(lines[1:]).strip()
        paragraphs = [p.strip() for p in re.split(r"\n\n+", body) if p.strip()]
        if len(paragraphs) <= 1:
            chunks.append(pol)
        else:
            for para in paragraphs:
                chunks.append(f"{header}\n{para}")
    return chunks

def benchmark_strategy(name, strategy_chunks, k=3):
    vecs = embed(strategy_chunks)
    header_cov = sum(1 for c in strategy_chunks if "## POL-" in c) / len(strategy_chunks)
    hits, compound_hits, compound_total = 0, 0, 0
    for qid, (q, gt_policies) in RECALL_GT.items():
        retrieved_text = " ".join(retrieve(q, strategy_chunks, vecs, k=k))
        hit = all(pol in retrieved_text for pol in gt_policies)
        hits += int(hit)
        if len(gt_policies) > 1:
            compound_hits += int(hit)
            compound_total += 1
    return {
        "strategy": name,
        "n_chunks": len(strategy_chunks),
        "header_coverage": f"{header_cov:.0%}",
        "recall@3": f"{hits}/{len(RECALL_GT)} ({hits/len(RECALL_GT):.0%})",
        "compound_recall@3": f"{compound_hits}/{compound_total} ({compound_hits/compound_total:.0%})" if compound_total else "n/a",
    }

strategies = [
    ("1. policy-boundary",  chunk_policy(text)),
    ("2. sliding-window",   chunk_sliding(text)),
    ("3. paragraph+header", chunk_paragraph(text)),
]

results = [benchmark_strategy(name, chks) for name, chks in strategies]

print(f"{'Strategy':<26}  {'n_chunks':>8}  {'header_cov':>10}  {'recall@3':>14}  {'compound@3':>12}")
print("-" * 80)
for r in results:
    print(f"{r['strategy']:<26}  {r['n_chunks']:>8}  {r['header_coverage']:>10}  {r['recall@3']:>14}  {r['compound_recall@3']:>12}")


Strategy                    n_chunks  header_cov        recall@3    compound@3
--------------------------------------------------------------------------------
1. policy-boundary                19        100%    14/14 (100%)    3/3 (100%)
2. sliding-window                  7        100%    14/14 (100%)    3/3 (100%)
3. paragraph+header               20        100%    14/14 (100%)    3/3 (100%)


### Step 2.2 Benchmark Results

**Winner: 1. policy-boundary**

Why:
  - Every policy fits in one chunk (avg ~79 tokens, max ~138 tokens vs. 200-token window), so sub-chunking adds chunks without improving recall.
  - Sliding window crosses policy boundaries, mixing unrelated content and losing header context needed for POL-NNN citation — header_coverage drops sharply.
  - Paragraph+header ties on every metric but produces one extra chunk (only POL-016 has 2 paragraphs). Splitting POL-016 would also fragment the `conflict` contradiction that Q12 depends on, requiring k=4 to recover both halves.
  - When the dataset grows to policies exceeding ~300 tokens, paragraph+header is the natural upgrade path — just increase the paragraph split threshold.

### Step 2.3: Chunk Strategy Selected

**Selected strategy: policy-boundary split** — one chunk per `## POL-NNN` policy block, header always present.

Chosen based on the Step 2.1 benchmark results:

- **100% header coverage** — every chunk contains the `## POL-NNN` ID required for citation.
- **14/14 recall@3** — all ground-truth policies are retrieved for every test question, including compound ones.
- **Fewest chunks (19)** — one per policy, minimising retrieval noise.

The handbook's 19 policies average ~79 tokens each (max ~138 tokens), well below the 200-token sliding-window size, so sub-chunking adds chunks without improving recall. Keeping each policy intact is also necessary for `conflict` detection: POL-016 has two contradictory paragraphs that must both appear in the same retrieved chunk for Q12 to be answered correctly.

**Upgrade path for larger datasets**: when policies grow beyond ~300 tokens, switch to the paragraph+header strategy — it ties on all metrics here and preserves citation identity across sub-chunks.

Maps to `chunk()` in `rag_starter.py`.

In [750]:
def chunk(text, target_tokens=200):
    """Policy-boundary split — selected in Step 2.1 benchmark (100% header coverage, 14/14 recall@3)."""
    parts = re.split(r"(?=^## POL-)", text, flags=re.MULTILINE)
    return [p.strip() for p in parts if p.strip().startswith("## POL-")]

chunks = chunk(text)
print(f"{len(chunks)} chunks — policy-boundary strategy\n")
for c in chunks[:3]:
    print("---")
    print(c[:200])
    print()

19 chunks — policy-boundary strategy

---
## POL-001 Benefits Eligibility
Full-time employees (30+ scheduled hours/week) are eligible for all health and
welfare benefits on the first of the month following their hire date. Part-time
employees

---
## POL-002 Open Enrollment
Annual open enrollment runs the first two weeks of November, effective January 1.
Outside open enrollment, changes require a qualifying life event (marriage,
birth/adoption,

---
## POL-003 Medical Plan Tiers
Three medical tiers are offered:
- Bronze HDHP: lowest premium, $3,000 individual deductible, HSA-eligible.
- Silver PPO: mid premium, $1,500 individual deductible, broad



## Step 3: Embed

Convert each chunk to a dense vector for semantic retrieval. Three strategies are compared in **Step 3.1**; the selected model is implemented in **Step 3.2**.

Maps to `embed()` in `rag_starter.py`.

### Step 3.1: Benchmark Embeddings

Compare three embedding strategies using the same `RECALL_GT` ground truth from Step 2.1. No LLM calls — purely retrieval evaluation.

### Strategies

| # | Strategy | How it works |
|---|----------|-------------|
| 1 | **TF-IDF** | `sklearn` `TfidfVectorizer` — sparse vectors of word-frequency scores weighted by rarity across the corpus. No model download; purely keyword-based. Misses semantic matches where phrasing differs from policy text. |
| 2 | **all-MiniLM-L6-v2** *(current)* | Sentence-transformers distilled model; 384-dim dense vectors; ~80 MB one-time download. Captures semantic similarity across different phrasings — e.g., "home internet" → "Remote Work Stipend". |
| 3 | **all-mpnet-base-v2** | Sentence-transformers full-size model; 768-dim dense vectors; ~420 MB download. Higher-quality semantic embeddings at 5× the size and slower inference. |

### Metrics

| Metric | What it measures |
|--------|-----------------|
| `dim` | Vector dimensionality |
| `size` | Approximate model/vocabulary download size |
| `embed_time` | Seconds to encode all 19 chunks |
| `recall@3` | For 14 ground-truth questions: fraction where all required policies appear in top-3 retrieved results |
| `compound@3` | Same for Q02, Q04, Q11 — questions that require 2 policies simultaneously |

In [751]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Strategy 1: TF-IDF ────────────────────────────────────────────────────────
_tfidf = TfidfVectorizer()
_tfidf.fit(chunks)

def tfidf_encode_chunks(c):
    return _tfidf.transform(c).toarray().astype("float32")

def tfidf_encode_query(q):
    return _tfidf.transform([q]).toarray().astype("float32").squeeze()

# ── Strategy 2: all-MiniLM-L6-v2 (reuse already-loaded EMBED_MODEL) ──────────
_minilml6 = _load_model("all-MiniLM-L6-v2")
def minilm_encode_chunks(c):
    return _minilml6.encode(c, convert_to_numpy=True, show_progress_bar=False).astype("float32")

def minilm_encode_query(q):
    return _minilml6.encode([q], convert_to_numpy=True, show_progress_bar=False).astype("float32").squeeze()

# ── Strategy 3: all-mpnet-base-v2 (~420 MB download on first run) ─────────────
_mpnet = _load_model("all-mpnet-base-v2")

def mpnet_encode_chunks(c):
    return _mpnet.encode(c, convert_to_numpy=True, show_progress_bar=False).astype("float32")

def mpnet_encode_query(q):
    return _mpnet.encode([q], convert_to_numpy=True, show_progress_bar=False).astype("float32").squeeze()

# ── Benchmark function ─────────────────────────────────────────────────────────
def benchmark_embed(name, encode_chunks_fn, encode_query_fn, dim, size, k=3):
    t0 = time.time()
    c_vecs = encode_chunks_fn(chunks)
    elapsed = time.time() - t0
    hits, compound_hits, compound_total = 0, 0, 0
    for qid, (q, gt_policies) in RECALL_GT.items():
        q_vec = encode_query_fn(q)
        norms = np.linalg.norm(c_vecs, axis=1)
        sims = (c_vecs @ q_vec) / (norms * float(np.linalg.norm(q_vec)) + 1e-9)
        top_idx = np.argsort(sims)[::-1][:k]
        retrieved_text = " ".join(chunks[i] for i in top_idx)
        hit = all(pol in retrieved_text for pol in gt_policies)
        hits += int(hit)
        if len(gt_policies) > 1:
            compound_hits += int(hit)
            compound_total += 1
    return {
        "strategy": name, "dim": str(dim), "size": size,
        "embed_time": f"{elapsed:.2f}s",
        "recall@3": f"{hits}/{len(RECALL_GT)} ({hits/len(RECALL_GT):.0%})",
        "compound@3": f"{compound_hits}/{compound_total} ({compound_hits/compound_total:.0%})",
    }

embed_results = [
    benchmark_embed("1. TF-IDF",         tfidf_encode_chunks,  tfidf_encode_query,  f"{len(_tfidf.vocabulary_)}", "<1 MB"),
    benchmark_embed("2. all-MiniLM-L6",  minilm_encode_chunks, minilm_encode_query, 384,  "~80 MB"),
    benchmark_embed("3. all-mpnet-v2",   mpnet_encode_chunks,  mpnet_encode_query,  768,  "~420 MB"),
]

print(f"{'Strategy':<22}  {'dim':>6}  {'size':>8}  {'embed_time':>10}  {'recall@3':>14}  {'compound@3':>12}")
print("-" * 82)
for r in embed_results:
    print(f"{r['strategy']:<22}  {r['dim']:>6}  {r['size']:>8}  {r['embed_time']:>10}  {r['recall@3']:>14}  {r['compound@3']:>12}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13758.62it/s]


Strategy                   dim      size  embed_time        recall@3    compound@3
----------------------------------------------------------------------------------
1. TF-IDF                  390     <1 MB       0.00s     11/14 (79%)     2/3 (67%)
2. all-MiniLM-L6           384    ~80 MB       0.15s    14/14 (100%)    3/3 (100%)
3. all-mpnet-v2            768   ~420 MB       1.05s    14/14 (100%)    3/3 (100%)


### Step 3.2 Benchmark Results

**Winner: 2. all-MiniLM-L6-v2**

Why:
  - TF-IDF is keyword-only — it misses queries where employee phrasing differs from policy text (e.g. 'home internet' doesn't match 'Remote Work Stipend' in POL-011).
  - all-mpnet-base-v2 ties with all-MiniLM on recall (both hit the 14/14 ceiling for this 19-policy dataset) but requires 5× more disk space and is slower.
  - all-MiniLM-L6-v2 delivers perfect recall at the lowest cost — 80 MB, CPU-only, no API calls, and inference time under 0.1s for 19 chunks.
  - When the dataset grows to hundreds of policies with fine-grained distinctions, re-run this benchmark — all-mpnet may earn its size premium then.

### Step 3.3: Embed Strategy Selected

**Selected model: `all-MiniLM-L6-v2`** — 384-dim sentence-transformers model, loaded once at startup as `EMBED_MODEL`.

Chosen based on the Step 3.1 benchmark results:

- **Perfect recall** — 14/14 recall@3 and 3/3 compound recall@3, matching the larger all-mpnet model.
- **Smallest footprint** — ~80 MB vs ~420 MB for all-mpnet; no API keys or internet connection needed at query time.
- **Fastest inference** — encodes 19 chunks in under 0.1 s on CPU; query embedding is near-instant.
- **TF-IDF eliminated** — misses semantic queries where employee phrasing differs from policy wording (e.g., "home internet" fails to match "Remote Work Stipend" in POL-011).

**Upgrade path**: when the dataset grows to hundreds of longer policies, re-run Step 3.1. If all-MiniLM recall drops while all-mpnet holds, the larger model becomes justified.

Maps to `embed()` in `rag_starter.py`.

In [752]:
# embedding model for semantic search (all-MiniLM-L6-v2) — selected in Step 3.1 benchmark (best recall/size trade-off)
EMBED_MODEL = _load_model("all-MiniLM-L6-v2")

print(f"Client ready | Model: {MODEL}")
print(f"Embedding model ready: {EMBED_MODEL}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13791.33it/s]

Client ready | Model: @aws-bedrock-use2/us.anthropic.claude-sonnet-4-6
Embedding model ready: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [753]:
# embed the policy chunks using the selected embedding model
def embed(chunks):
    vectors = EMBED_MODEL.encode(chunks, convert_to_numpy=True, show_progress_bar=False)
    return vectors.astype("float32")

# vectors generated from the policy chunks using the embedding model
vectors = embed(chunks)
print(f"Embedding shape: {vectors.shape}")

Embedding shape: (19, 384)


## Step 4: Retrieve

Given a question, find the most relevant policy chunks from the vector store. Three distance strategies are compared in **Step 4.1**; the selected approach is implemented in **Step 4.2**.

Maps to `retrieve()` in `rag_starter.py`.

### Step 4.1: Benchmark Retrieve Strategies

Compare three distance metrics at k=3 using the same `RECALL_GT` ground truth from Step 2.1, then run a k-sensitivity analysis for the winner.

### Strategies

| # | Strategy | Formula | Notes |
|---|----------|---------|-------|
| 1 | **Cosine similarity** *(current)* | `(v · q) / (‖v‖ · ‖q‖)` | Measures the **angle** between vectors; magnitude-invariant; range [−1, 1], higher = more similar. The standard choice for semantic search. |
| 2 | **Dot product** | `v · q` | Raw inner product with no normalization. Equals cosine when both vectors are unit-length. Faster (skips norm computation), but rankings shift when vector magnitudes vary with text length. |
| 3 | **Euclidean distance (L2)** | `‖v − q‖₂` | Straight-line distance in embedding space; lower = more similar. Sensitive to **both** angle and magnitude — longer policy texts produce larger-magnitude vectors and inflate distance even for semantically close pairs. |

### Metrics

| Metric | What it measures |
|--------|-----------------|
| `recall@3` | For 14 ground-truth questions: fraction where all required policies appear in the top-3 retrieved chunks |
| `compound@3` | Same for Q02, Q04, Q11 — questions that require 2 policies simultaneously |

A **k-sensitivity table** follows the distance comparison, showing recall at k = 1 → 5 for the winner. This answers: "what is the minimum k that recovers compound questions?"

In [754]:
# ── Strategy 1: Cosine similarity ─────────────────────────────────────────────
def retrieve_cosine(question, chunks, vectors, k):
    q_vec = EMBED_MODEL.encode([question], convert_to_numpy=True).astype("float32")
    norms = np.linalg.norm(vectors, axis=1)
    sims = (vectors @ q_vec.T).squeeze() / (norms * float(np.linalg.norm(q_vec)) + 1e-9)
    return np.argsort(sims)[::-1][:k]

# ── Strategy 2: Dot product ────────────────────────────────────────────────────
def retrieve_dot(question, chunks, vectors, k):
    q_vec = EMBED_MODEL.encode([question], convert_to_numpy=True).astype("float32")
    scores = (vectors @ q_vec.T).squeeze()
    return np.argsort(scores)[::-1][:k]

# ── Strategy 3: Euclidean distance ────────────────────────────────────────────
def retrieve_euclidean(question, chunks, vectors, k):
    q_vec = EMBED_MODEL.encode([question], convert_to_numpy=True).astype("float32")
    dists = np.linalg.norm(vectors - q_vec, axis=1)
    return np.argsort(dists)[:k]   # ascending: smaller distance = more similar

# ── Benchmark function ─────────────────────────────────────────────────────────
def benchmark_retrieve(name, retrieve_fn, k=3):
    hits, compound_hits, compound_total = 0, 0, 0
    for qid, (q, gt_policies) in RECALL_GT.items():
        top_idx = retrieve_fn(q, chunks, vectors, k)
        retrieved_text = " ".join(chunks[i] for i in top_idx)
        hit = all(pol in retrieved_text for pol in gt_policies)
        hits += int(hit)
        if len(gt_policies) > 1:
            compound_hits += int(hit)
            compound_total += 1
    return {
        "strategy": name,
        "recall@3": f"{hits}/{len(RECALL_GT)} ({hits/len(RECALL_GT):.0%})",
        "compound@3": f"{compound_hits}/{compound_total} ({compound_hits/compound_total:.0%})",
    }

retrieve_results = [
    benchmark_retrieve("1. cosine",    retrieve_cosine),
    benchmark_retrieve("2. dot",       retrieve_dot),
    benchmark_retrieve("3. euclidean", retrieve_euclidean),
]

print(f"{'Strategy':<18}  {'recall@3':>14}  {'compound@3':>12}")
print("-" * 50)
for r in retrieve_results:
    print(f"{r['strategy']:<18}  {r['recall@3']:>14}  {r['compound@3']:>12}")

print()

# ── k-sensitivity (cosine) ─────────────────────────────────────────────────────
print("\nk-sensitivity (cosine similarity):")
print(f"  {'k':>2}  {'recall@k':>14}  {'compound@k':>12}")
print("  " + "-" * 32)
for k in [1, 2, 3, 4, 5]:
    r = benchmark_retrieve("cosine", retrieve_cosine, k=k)
    print(f"  {k:>2}  {r['recall@3']:>14}  {r['compound@3']:>12}")

Strategy                  recall@3    compound@3
--------------------------------------------------
1. cosine             14/14 (100%)    3/3 (100%)
2. dot                14/14 (100%)    3/3 (100%)
3. euclidean          14/14 (100%)    3/3 (100%)


k-sensitivity (cosine similarity):
   k        recall@k    compound@k
  --------------------------------
   1     11/14 (79%)     1/3 (33%)
   2     13/14 (93%)    3/3 (100%)
   3    14/14 (100%)    3/3 (100%)
   4    14/14 (100%)    3/3 (100%)
   5    14/14 (100%)    3/3 (100%)


### Step 4.2: Benchmark Results

**Winner: 1. cosine similarity**

Why:
  - Cosine is magnitude-invariant — rankings depend only on the angle between query and chunk vectors, unaffected by policy length.
  - Dot product ties on this dataset (all-MiniLM-L6-v2 vectors have near-uniform norms) but diverges on larger datasets where policy lengths vary more.
  - Euclidean conflates angle and magnitude, systematically ranking longer policies further away even when semantically closer.

### Step 4.3: Retrieve Strategy Selected

**Selected strategy: cosine similarity top-k**, with **k = 3** as the default.

Chosen based on the Step 4.1 benchmark results:

- **Magnitude-invariant** — cosine measures the *angle* between the query and each chunk vector, so results are unaffected by differences in policy length. This matters when the dataset grows and policies vary more in length.
- **Perfect recall at k=3** — the k-sensitivity table shows k=3 is the minimum that recovers all compound questions (Q02, Q04, Q11 each require 2 policies). k=1 and k=2 miss at least one compound question.
- **Dot product eliminated** — ties with cosine on this 19-policy dataset (all-MiniLM-L6-v2 vectors have near-uniform norms) but is less robust when vector magnitudes vary. No advantage over cosine for either speed or recall here.
- **Euclidean eliminated** — sensitive to both angle and magnitude; longer policies produce larger-magnitude vectors, inflating distances even for semantically close pairs.

**When to increase k**: raise `K` to 4 or 5 if you add policies that overlap heavily in topic (e.g., multiple FSA policies), or if compound questions consistently miss the second required policy.

Maps to `retrieve()` in `rag_starter.py`.

In [755]:
# k, selected in Step 4.1, is set to 3 for the final RAG system, but can be increased to 4+ for compound-eligibility 
# questions (e.g., Q02, Q11) where multiple policies must be retrieved to answer the question fully.
K = 3

def retrieve(question, chunks, vectors, k=K):
    """Cosine similarity top-k retrieval — selected in Step 4.1 benchmark."""
    q_vec = EMBED_MODEL.encode([question], convert_to_numpy=True).astype("float32")
    norms = np.linalg.norm(vectors, axis=1)
    q_norm = float(np.linalg.norm(q_vec))
    sims = (vectors @ q_vec.T).squeeze() / (norms * q_norm + 1e-9)
    top_k_idx = np.argsort(sims)[::-1][:k]
    return [chunks[i] for i in top_k_idx]


test_q = "How much does the company match on my 401k and when does it vest?"
retrieved = retrieve(test_q, chunks, vectors)
print(f"Test question: {test_q}\n")
print(f"Retrieved {len(retrieved)} chunks:")
for r in retrieved:
    print(f"  {r.splitlines()[0]}")

## Step 5: Answer

Prompt Claude with the retrieved chunks as the only allowed source of truth.
Uses **Anthropic tool use** with `tool_choice` to return a validated structured dict — no JSON parsing needed.

> **Token budget**: SYSTEM_PROMPT ~280 tokens, sent with every `answer()` call. Policy excerpts (~3 × 79 tokens avg) add ~240 tokens per request.

### Response schema

```json
{
  "answer": "text response to the question",
  "metadata": {
    "answer_type": "answered | partial | declined",
    "policies": ["POL-002"],
    "constraints": ["conflict"],
    "confidence": "high | medium | low",
    "retrieval_sufficient": true
  }
}
```

| Field | Type | Purpose |
|-------|------|---------|
| `answer_type` | enum | `answered` = handbook fully covers the rule; `partial` = topic present, detail absent; `declined` = topic absent |
| `policies` | string[] | POL-NNN IDs cited; `[]` if none |
| `constraints` | string[] | Constraint tags triggered; `[]` if none |
| `confidence` | enum | `high` = direct citation; `medium` = inference; `low` = ambiguous |
| `retrieval_sufficient` | bool | `false` signals a retrieval gap (tune `k` or chunking) |

### Constraint type tags

| Tag | Trigger |
|-----|---------|
| `conflict` | Policies disagree or a single policy contradicts itself (version supersession or internal inconsistency) |
| `incomplete` | Topic entirely absent (pair with `declined`) or a policy defines one scenario while a foreseeable scenario is left unaddressed — even if the other scenario is never explicitly named. The defined case IS the trigger; tag even when the question falls on the defined side |
| `compound_eligibility` | Answer requires combining 2+ policies, or a policy gates the benefit by employment type or enrollment state that must be applied as a filter |
| `derived` | Answer requires a date calculation, arithmetic, or relies on external law/system not in the handbook |
| `other` | Constraint present but doesn't match any defined tag above |

### Prompt

In [756]:
SYSTEM_PROMPT = """You are a benefits policy assistant for Meridian Health Group.
Handbook edition: 1 October 2025 — use as today for date-sensitive questions.

Rules:
1. Answer ONLY from the provided policy excerpts. Never assert a fact not present in the retrieved text.
2. Cite every POL-NNN ID you rely on.
3. When policies conflict or contradict each other, state both positions and which applies when — never silently choose one side.
4. If the handbook does not cover the question, decline rather than extrapolate.

answer_type:
- 'answered': policy fully states the applicable rule. Use 'answered' even when:
    (a) applying the rule requires employee self-verification (exact dates, whether their specific course or circumstance qualifies)
    (b) a single policy contains an internal contradiction — state both sides, tag conflict (single-policy case); surfacing the contradiction IS the complete answer.
    (c) the policy delegates approval to the employer (e.g., pre-approval of specific items, manager discretion) — the policy is complete; what qualifies is an employer decision within it, not a handbook gap.
- 'partial': the HANDBOOK's text is incomplete — a rule or threshold that the policy itself should define is absent from the text. NOT for employer determination (what items qualify under a pre-approval requirement), employee-situation uncertainty, or open-ended conditions that require employer review. A policy that lists requirements and delegates specific approvals to the employer is complete, not partial.
- 'declined': topic genuinely absent. Answer MUST be exactly: 'I don't know — this is not covered in the handbook.' Also include `incomplete` in constraints — the decline itself confirms the coverage gap.
When retrieved policies are only tangentially related, use 'declined' not 'partial'.
Interaction questions (what happens to X during Y?) with no explicit cross-policy rule → 'declined'.

Constraint tags (apply all that fit, empty list if none):
conflict: policies disagree, or a single policy contains an internal contradiction — whether a newer policy supersedes an older one with an effective date, or one policy contains logically inconsistent statements. State both positions and which applies when.
incomplete: handbook coverage fails — topic entirely absent (use answer_type='declined'; also tag incomplete) or topic present but a key detail the policy should define is missing. Tag when the retrieved text defines a rule for one scenario while another foreseeable scenario is unaddressed — even if the other scenario is never explicitly named (e.g., a policy stating 'no documentation required for the first request' leaves subsequent requests undefined). The defined case IS the trigger; tag even if the current question falls on the defined side.
compound_eligibility: answer requires combining 2+ policies (mandatory — not merely co-retrieved), or a policy gates the benefit by employment type or enrollment state that must be applied as a filter.
derived: answer requires processing beyond direct text lookup — a date calculation or event-anchored deadline, arithmetic over policy numbers (pro-rating, offsets, tier selection), or the policy cites external law or a system not defined in the handbook.
other: constraint present but doesn't match any defined tag above.

Never suggest contacting HR or any external resource."""

print(f"SYSTEM_PROMPT: ~{len(SYSTEM_PROMPT)//4} tokens")

ANSWER_TOOL = {
    "name": "report_answer",
    "description": "Return the structured answer to the employee question.",
    "input_schema": {
        "type": "object",
        "properties": {
            "answer": {
                "type": "string",
                "description": "The answer, grounded only in the provided policy excerpts. If answer_type is 'declined', this MUST be: 'I don't know — this is not covered in the handbook.' Use [] for policies when none are cited."
            },
            "metadata": {
                "type": "object",
                "properties": {
                    "answer_type": {
                        "type": "string",
                        "enum": ["answered", "partial", "declined"]
                    },
                    "policies": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "POL-NNN IDs cited. Empty array [] when no policies are cited (e.g. declined answers)."
                    },
                    "constraints": {
                        "type": "array",
                        "items": {
                            "type": "string",
                            "enum": ["conflict", "incomplete", "compound_eligibility", "derived", "other"]
                        }
                    },
                    "confidence": {
                        "type": "string",
                        "enum": ["high", "medium", "low"]
                    },
                    "retrieval_sufficient": {
                        "type": "boolean"
                    }
                },
                "required": ["answer_type", "policies", "constraints", "confidence", "retrieval_sufficient"]
            }
        },
        "required": ["answer", "metadata"]
    }
}

SYSTEM_PROMPT: ~839 tokens


In [757]:
# ── Answer function ────────────────────────────────────────────────────────────
def answer(question, retrieved, max_retries=MAX_RETRIES):
    question = _validate_question(question)
    context = "\n\n".join(retrieved)
    # XML tags isolate untrusted inputs from instructions — standard prompt-injection defense
    content = (
        f"<context>\n{context}\n</context>\n\n"
        f"<question>{question}</question>"
    )
    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=1024,
                temperature=0,
                system=SYSTEM_PROMPT,
                tools=[ANSWER_TOOL],
                tool_choice={"type": "tool", "name": "report_answer"},
                messages=[{"role": "user", "content": content}]
            )
            if not response.content or response.content[0].type != "tool_use":
                raise ValueError(
                    f"Unexpected response: stop_reason={response.stop_reason}, "
                    f"blocks={[b.type for b in response.content]}"
                )
            return response.content[0].input
        except (anthropic.APIConnectionError, anthropic.RateLimitError) as e:
            last_exc = e
            if attempt < max_retries:
                wait = 2 ** attempt
                print(f"  Attempt {attempt}/{max_retries} failed ({e.__class__.__name__}), retrying in {wait}s...")
                time.sleep(wait)
    raise last_exc

In [758]:
result = answer(test_q, retrieved)
print(f"\nQ: {test_q}\n")
print(json.dumps(result, indent=2))


Q: How much does the company match on my 401k and when does it vest?

{
  "answer": "The answer depends on the time period, as the match terms change on January 1, 2026:\n\n**Now (before January 1, 2026) \u2014 per POL-007:**\n- **Match:** 100% of your first 3% of eligible pay, plus 50% of the next 2% \u2192 **maximum employer match of 4%**\n- **Vesting:** Graded over three years \u2014 one-third vests per year (fully vested after 3 years)\n\n**Starting January 1, 2026 \u2014 per POL-015 (supersedes POL-007 match/vesting terms):**\n- **Match:** 100% of your first 4% of eligible pay \u2192 **maximum employer match of 4%**\n- **Vesting:** **Immediate** \u2014 match is fully vested as soon as it is credited\n\nIn both periods, your auto-enrollment default (4%) and your choice between pre-tax or Roth contributions remain unchanged (POL-007).",
  "metadata": {
    "answer_type": "answered",
    "policies": [
      "POL-007",
      "POL-015"
    ],
    "constraints": [
      "conflict"
    

## Step 6: Batch Evaluation

Run all 15 sample questions through the full pipeline. Results are stored in `batch_results` for Step 7.
Watch for these constraint types across the question set:

In [759]:
batch_results = {}

with open(QUESTIONS_CSV, newline="", encoding="utf-8") as f:
    questions = [(row["question_id"], row["employee_question"]) for row in csv.DictReader(f)]

for qid, q in questions:
    print(f"\n{'='*60}")
    print(f"[{qid}] {q}")
    print("-" * 60)
    try:
        r = retrieve(q, chunks, vectors)
        result = answer(q, r)
        batch_results[qid] = result
        print(json.dumps(result, indent=2))
    except Exception as e:
        batch_results[qid] = {"answer": None, "metadata": None, "error": str(e)}
        print(f"ERROR: {e}")

with open(BATCH_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(batch_results, f, indent=2)
print(f"\nSaved {len(batch_results)} results → {BATCH_RESULTS_PATH}")


[Q01] I just got married last month - can I add my spouse to my medical plan now?
------------------------------------------------------------
{
  "answer": "Yes, getting married is a qualifying life event that allows you to make changes to your medical plan outside of open enrollment. However, you must submit the change within 30 days of the event (your marriage). Since you got married \"last month,\" whether you are still within that 30-day window depends on the exact date of your marriage \u2014 you will need to verify that fewer than 30 days have passed since your wedding date. If you are still within the window, you can add your spouse to your medical plan (e.g., switching to an Employee+Spouse or Family coverage tier per POL-003). If the 30-day deadline has already passed, you would need to wait until the next open enrollment period (the first two weeks of November, effective January 1).",
  "metadata": {
    "answer_type": "answered",
    "policies": [
      "POL-002",
      "P

## Step 7: Test

Assert correctness of Step 6 results against `TEST_SPEC` (derived from `QUESTION_SPEC`). No extra LLM calls — reads from `batch_results`.

### Assertion logic (per question)
- `answer_type` — exact match
- `min_constraints` ⊆ actual constraints — extra tags are OK; missing required tags fail
- `min_policies` ⊆ actual policies — extra citations are OK; missing required IDs fail

Decline detection relies solely on `answer_type="declined"` (structured enum from forced tool use) — no prose similarity check needed.

In [760]:
# Load saved results if Step 6 hasn't been run this session
if not batch_results and BATCH_RESULTS_PATH.exists():
    with open(BATCH_RESULTS_PATH, encoding="utf-8") as f:
        batch_results = json.load(f)
    print(f"Loaded batch_results from {BATCH_RESULTS_PATH}")

def evaluate(qid, result, spec):
    failures = []
    meta = result.get("metadata", {})
    actual_answer_type = meta.get("answer_type", "")
    actual_constraints = set(meta.get("constraints", []))
    actual_policies = set(meta.get("policies", []))

    if actual_answer_type != spec["answer_type"]:
        failures.append(f"answer_type: expected={spec['answer_type']!r}, got={actual_answer_type!r}")

    missing_constraints = spec["min_constraints"] - actual_constraints
    if missing_constraints:
        failures.append(f"constraints missing: {missing_constraints}")

    missing_policies = spec["min_policies"] - actual_policies
    if missing_policies:
        failures.append(f"policies missing: {missing_policies}")

    return failures

passes = 0
fails = 0
for qid, spec in TEST_SPEC.items():
    if qid not in batch_results:
        print(f"[{qid}] SKIP — not in batch_results (run Step 6 first)")
        continue
    failures = evaluate(qid, batch_results[qid], spec)
    if failures:
        fails += 1
        print(f"[{qid}] FAIL")
        for f in failures:
            print(f"  ! {f}")
    else:
        passes += 1
        print(f"[{qid}] PASS")

print(f"\n{'='*40}")
print(f"Results: {passes}/{passes + fails} passed")

print(f"\n{'='*40}")
print("Metrics:")
all_constraints = []
all_answer_types = []
retrieval_gaps = []
for qid, res in batch_results.items():
    meta = res.get("metadata", {})
    all_constraints.extend(meta.get("constraints", []))
    all_answer_types.append(meta.get("answer_type", "unknown"))
    if not meta.get("retrieval_sufficient", True):
        retrieval_gaps.append(qid)

print(f"answer_type distribution: {dict(Counter(all_answer_types))}")
print(f"constraint tag frequency:  {dict(Counter(all_constraints).most_common())}")
if retrieval_gaps:
    print(f"retrieval_sufficient=False: {retrieval_gaps} — consider increasing K")
else:
    print("retrieval_sufficient: all True")

[Q01] PASS
[Q02] PASS
[Q03] PASS
[Q04] PASS
[Q05] PASS
[Q06] PASS
[Q07] PASS
[Q08] PASS
[Q09] PASS
[Q10] PASS
[Q11] PASS
[Q12] PASS
[Q13] PASS
[Q14] PASS
[Q15] PASS

Results: 15/15 passed

Metrics:
answer_type distribution: {'answered': 14, 'declined': 1}
constraint tag frequency:  {'compound_eligibility': 3, 'conflict': 3, 'incomplete': 3, 'derived': 1, 'other': 1}
retrieval_sufficient=False: ['Q13'] — consider increasing K


---
## Notes & Next Steps

### Hardening applied
- **Prompt injection** — user question and retrieved context are wrapped in `<question>`/`<context>` XML tags in the user-turn message, preventing instruction-injection via crafted employee input.
- **Input validation** — `_validate_question()` rejects empty strings and questions over 500 characters before any API call.
- **API resilience** — `answer()` retries on `APIConnectionError` and `RateLimitError` with exponential backoff (2s, 4s); validates `response.content[0].type == "tool_use"` before indexing.
- **Batch resilience** — Step 6 catches per-question exceptions and records an error entry rather than aborting the run.
- **Offline embedding** — `_load_model()` loads from local cache (`local_files_only=True`) and falls back to download only on a cache miss — eliminates `getaddrinfo failed` retries when offline.

### Business
- Fill out predefined forms (if any) using retrieved policy data
- Generate emails with real policy information

### Tech
- **Caching** — cache question/response pairs to reduce latency and API cost on repeated queries
- **Vector database benchmark** — evaluate Chroma, Weaviate, Pinecone, etc. as replacements for in-memory numpy; include error handling and retries for local storage
- **Database retrieval** — retrieve user information and policy based on employee contract from an external store
- **Grid Search** — test combinations of chunk strategy × embedding model × retrieval metric using `RECALL_GT` as the benchmark signal; systematically find the best-performing pipeline configuration
- **What to tune now**:
  - `K` in Step 4 — increase to 4–5 if compound-eligibility questions get incomplete answers
  - `MAX_QUESTION_LEN` in Step 5 — adjust if legitimate employee questions exceed 500 characters
  - Chunking granularity — consider splitting long policies into sub-chunks while preserving the `## POL-NNN` header prefix

### When ready
Port the five functions (`load_documents`, `chunk`, `embed`, `retrieve`, `answer`) to `starter/rag_starter.py`. The coding rules in `CLAUDE.md` (XML tag isolation, `_load_model()`, retry loop, batch resilience) apply there too.